# Exploration du pipeline RAG Assurance

Ce notebook déroule le pipeline étape par étape : chargement → chunking → embeddings → retrieval → génération.

Prérequis : `pip install -r ../requirements.txt` et `ANTHROPIC_API_KEY` renseignée dans `.env`.

In [ ]:
import sys
sys.path.insert(0, "..")

from app.config import settings
from app.embeddings import load_documents, split_documents, ingest, get_vectorstore
from app.rag_chain import retrieve, format_context, answer

settings

## 1. Chargement des documents

In [ ]:
docs = load_documents(settings.raw_dir)
print(f"{len(docs)} documents chargés")
print(docs[0].metadata)
print(docs[0].page_content[:400])

## 2. Chunking

Le `RecursiveCharacterTextSplitter` coupe en priorité sur les sauts de paragraphe, ce qui préserve la structure article/alinéa des conditions générales.

In [ ]:
chunks = split_documents(docs)
print(f"{len(chunks)} chunks")
lengths = [len(c.page_content) for c in chunks]
print(f"taille min/moy/max : {min(lengths)} / {sum(lengths)//len(lengths)} / {max(lengths)}")
print(chunks[0].page_content)

## 3. Indexation dans ChromaDB

In [ ]:
n = ingest()
print(f"{n} chunks indexés")
get_vectorstore()._collection.count()

## 4. Retrieval — vérifier la pertinence AVANT de générer

Si le retrieval est mauvais, aucun prompt ne sauvera la réponse. C'est l'étape à instrumenter en premier.

In [ ]:
question = "Quelles sont les exclusions de garantie pour un sinistre incendie ?"
hits = retrieve(question)
for h in hits:
    print(h.metadata.get("source"), "|", h.page_content[:150].replace("
", " "), "
")

In [ ]:
print(format_context(hits)[:1200])

## 5. Génération sourcée

In [ ]:
res = answer(question)
print(res.answer)

## 6. Évaluation sur un jeu de questions types

In [ ]:
questions = [
    "Quelles sont les exclusions de garantie pour un sinistre incendie ?",
    "Quel est le délai de déclaration pour un dommage matériel ?",
    "Résume les conditions de prise en charge d'une assistance à domicile.",
    "Quel est le montant de la franchise en cas de dégât des eaux ?",
    "Quelles pièces justificatives sont exigées pour un dossier de sinistre ?",
    "Le contrat couvre-t-il les dommages causés par un satellite ?",  # hors corpus : doit refuser
]

for q in questions:
    print("=" * 90)
    print("Q:", q)
    print(answer(q).answer)
    print()